In [ ]:
from pathlib import Path
from io import BytesIO
import json, csv, hashlib, math

PACKAGE_ROOT = next((p for p in (Path.cwd(), Path.cwd().parent)
                     if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if PACKAGE_ROOT is None:
    raise FileNotFoundError("Open this notebook from the package root or notebooks directory.")

SAVE_PDF = False

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import t
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from IPython.display import display, Image
plt.rcdefaults()

def load_npz(relative_path):
    with np.load(PACKAGE_ROOT / relative_path, allow_pickle=False) as archive:
        return {key: archive[key].copy() for key in archive.files}

def add_panel_labels(figure):
    panels = [axis for axis in figure.axes if axis.get_label() != "<colorbar>"]
    assert len(panels) == 6, "Unexpected panel count"
    for index, axis in enumerate(panels):
        axis.text(-0.16, 1.04, f"({chr(97 + index)})",
                  transform=axis.transAxes, fontsize=12, fontweight="bold",
                  ha="left", va="bottom", clip_on=False, gid="panel_label")
    figure.canvas.draw()
    renderer = figure.canvas.get_renderer()
    records = []
    for axis in panels:
        label = next(text for text in axis.texts if text.get_gid() == "panel_label")
        box = label.get_window_extent(renderer)
        for title in (axis.title, axis._left_title, axis._right_title):
            if title.get_text():
                assert not box.overlaps(title.get_window_extent(renderer)), "Panel label overlaps title"
        records.append({"label": label.get_text(), "axes_position": list(axis.get_position().bounds),
                        "label_bbox_pixels": list(box.bounds)})

def export_figure(figure, figure_id, tight=False):
    add_panel_labels(figure)
    options = {"bbox_inches": "tight"} if tight else {}
    preview = BytesIO()
    figure.savefig(preview, format="png", dpi=180, **options)
    display(Image(data=preview.getvalue()))
    if SAVE_PDF:
        output = PACKAGE_ROOT / "outputs"
        output.mkdir(exist_ok=True)
        figure.savefig(output / f"{figure_id}.pdf",
                       metadata={"CreationDate": None, "ModDate": None}, **options)
        print(f"Saved outputs/{figure_id}.pdf")


In [ ]:
MANIFEST=json.loads((PACKAGE_ROOT/"data/FigS3_FigS4_FigS5/manifest.json").read_text())
CONDITIONS = [dict(c) for c in MANIFEST["conditions"]]
LABEL_MAP = {}
for c in CONDITIONS:
    if not c["noise_enabled"]:
        LABEL_MAP[c["key"]] = {"title": "Noiseless\n", "label": "Noiseless"}
    elif c["key"] in ("main_y_015", "main_y_030"):
        probability = c["encoder_noise"][1]
        LABEL_MAP[c["key"]] = {"title": "Dephasing noise:\n" + rf"$p_x=p_y=0,\ p_z={probability:.2f}$",
                               "label": rf"Dephasing: $p_x=p_y=0,\ p_z={probability:.2f}$"}
    else:
        probability = c["encoder_noise"][0]
        LABEL_MAP[c["key"]] = {"title": c["title"], "label": rf"Depolarizing: $p_x=p_y=p_z={probability:.2f}$"}
    if True:
        c.update(LABEL_MAP[c["key"]])
CONDITION_BY_KEY = {c["key"]: c for c in CONDITIONS}

PRODUCTS={}
for item in MANIFEST["inputs"]:
    data=load_npz("data/FigS3_FigS4_FigS5/"+item["file"])
    PRODUCTS[item["key"]]={"amplitude":data["amplitude"],"bands":{b[0]:data["band_"+b[0]] for b in MANIFEST["bands"]}}
BANDS=MANIFEST["bands"]
SELECTED_FREQUENCIES=(1,5,6,13,19,21)
COMMON_END=500
LEGACY_SPECTRUM_STATES=tuple(range(0,200,10))+tuple(range(200,500,50))+(499,)
LEGACY_SELECTED_STATES=600
T975_DF19=2.093024054
PLOT_WARNINGS=[]
OUTPUT=None

In [ ]:
plt.rcParams.update({
    "font.family": "serif", "font.serif": ["DejaVu Serif"],
    "font.size": 10, "axes.titlesize": 11.5, "axes.labelsize": 11,
    "xtick.labelsize": 9.5, "ytick.labelsize": 9.5, "legend.fontsize": 8.5,
    "pdf.fonttype": 42, "ps.fonttype": 42, "axes.linewidth": 0.8,
})
def mean_ci95(values, axis=0):
    values = np.asarray(values, dtype=np.float64)
    return np.mean(values, axis=axis), T975_DF19 * np.std(values, axis=axis, ddof=1) / np.sqrt(values.shape[axis])

def style_axis(axis):
    axis.tick_params(direction="out", length=3.5, width=0.8)
    for spine in axis.spines.values():
        spine.set_linewidth(0.8)

def band_limits(products, end):
    lower, upper = 1.0, 1.0
    for key, *_ in BANDS:
        values = products["bands"][key][:, :end + 1]
        mean, half = mean_ci95(values)
        positive = mean - half
        positive = positive[positive > 0]
        if positive.size:
            lower = min(lower, float(np.min(positive)))
        upper = max(upper, float(np.max(mean + half)))
    return max(1e-7, lower / 1.5), upper * 1.15

def selected_amplitude_figure(keys, full_range, stem):
    figure, axes = plt.subplots(2, 3, figsize=(11.0, 7.4), sharex=True, sharey=True)
    maximum = max(float(np.max(MEAN_AMPLITUDES[key][:, list(SELECTED_FREQUENCIES)])) for key in keys)
    y_upper = max(0.023, maximum * 1.04)
    for axis, frequency in zip(axes.flat, SELECTED_FREQUENCIES):
        for key in keys:
            condition = CONDITION_BY_KEY[key]
            last = condition["epochs"] if full_range else LEGACY_SELECTED_STATES - 1
            values = MEAN_AMPLITUDES[key][:last + 1, frequency]
            axis.plot(np.arange(last + 1), values, color=condition["color"], linestyle="-" if len(keys) == 4 else condition["linestyle"],
                      linewidth=0.95, label=condition["label"])
        axis.set_title(f"Frequency = {frequency}")
        axis.set_xlim(0, 1600 if full_range else 599)
        axis.set_ylim(-0.001, y_upper)
        axis.set_xticks([0, 400, 800, 1200, 1600] if full_range else [0, 100, 200, 300, 400, 500, 599])
        style_axis(axis)
    for axis in axes[1]:
        axis.set_xlabel("Training iteration")
    for axis in axes[:, 0]:
        axis.set_ylabel("Amplitude")
    handles, labels = axes[0, 0].get_legend_handles_labels()
    figure.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.99), ncol=2 if len(keys) == 4 else 3, frameon=False)
    figure.subplots_adjust(left=0.09, right=0.985, bottom=0.09, top=0.86, hspace=0.32, wspace=0.15)
    save_show(figure, "paper_style", stem)

def plot_band_curves(full_range, stem):
    ends = {c["key"]: c["epochs"] if full_range else COMMON_END for c in CONDITIONS}
    limits = [band_limits(PRODUCTS[c["key"]], ends[c["key"]]) for c in CONDITIONS]
    ylim = (min(v[0] for v in limits), max(v[1] for v in limits))
    figure, axes = plt.subplots(2, 3, figsize=(11.5, 7.8), sharey=True)
    for axis, condition in zip(axes.flat, CONDITIONS):
        end = ends[condition["key"]]
        for key, label, _, _, color, linestyle in BANDS:
            mean, half = mean_ci95(PRODUCTS[condition["key"]]["bands"][key][:, :end + 1])
            axis.plot(np.arange(end + 1), mean, color=color, linestyle=linestyle, linewidth=1.2, label=label)
            axis.fill_between(np.arange(end + 1), np.maximum(mean - half, ylim[0]), mean + half, color=color, alpha=0.12, linewidth=0)
        axis.axhline(1, color="0.35", linestyle=(0, (3, 2)), linewidth=0.7)
        axis.set_yscale("log")
        axis.set_ylim(*ylim)
        axis.set_xlim(0, end)
        axis.set_xticks(np.linspace(0, end, 5, dtype=int))
        axis.set_title(condition["title"])
        axis.grid(alpha=0.16, which="both")
        style_axis(axis)
    for axis in axes[1]:
        axis.set_xlabel("Training iteration")
    for axis in axes[:, 0]:
        axis.set_ylabel(r"Relative band loss $L_B(n)/L_B(0)$")
    handles, labels = axes[0, 0].get_legend_handles_labels()
    figure.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.0), ncol=4, frameon=False)
    figure.subplots_adjust(left=0.085, right=0.985, bottom=0.11, top=0.88, hspace=0.40, wspace=0.15)
    save_show(figure, "frequency_loss", stem)
def save_show(figure,folder,stem):
    export_figure(figure,"FigS5",tight=True)
    plt.close(figure)

In [ ]:
plot_band_curves(False,"bands")